# 05 – Bullwhip Effect and Demand Spike Analysis

**Owner:** Bhumika KN. All sample data below is synthetic and intended only to demonstrate the methods.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analytics.spike_detection import detect_demand_spikes
from src.analytics.bullwhip import calculate_bullwhip_ratio, bullwhip_by_stage

## 1. Demand spike detection

In [ ]:
history = pd.read_csv(ROOT / 'data' / 'dummy_daily_demand.csv', parse_dates=['date'])
p1 = history[history['sku'] == 'P001'].copy()
spike_results = detect_demand_spikes(p1, demand_col='actual_demand', window=5, threshold=2.5, method='zscore')
display(spike_results)

In [ ]:
ax = spike_results.plot(x='date', y='actual_demand', marker='o', figsize=(10, 4), title='Demand history and detected spikes')
spikes = spike_results[spike_results['is_spike']]
if not spikes.empty:
    ax.scatter(spikes['date'], spikes['actual_demand'], marker='x', s=100, label='Detected spike')
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Bullwhip ratio for two aligned series

In [ ]:
customer_demand = [100, 105, 98, 110, 102, 108, 99, 112]
retailer_orders = [95, 120, 85, 135, 90, 125, 80, 140]
ratio = calculate_bullwhip_ratio(customer_demand, retailer_orders)
print('Bullwhip ratio:', round(ratio, 3))
print('Interpretation:', 'Order variance is amplified' if ratio > 1 else 'Order variance is not amplified')

## 3. Compare multiple supply-chain stages

In [ ]:
stages = []
periods = range(1, 9)
series_by_stage = {
    'customer_demand': customer_demand,
    'retailer_orders': retailer_orders,
    'wholesaler_orders': [90, 130, 75, 150, 85, 140, 70, 160],
    'manufacturer_orders': [80, 145, 65, 165, 75, 155, 60, 175],
}
for stage, values in series_by_stage.items():
    for period, quantity in zip(periods, values):
        stages.append({'period': period, 'stage': stage, 'quantity': quantity})
stage_df = pd.DataFrame(stages)
summary = bullwhip_by_stage(stage_df)
display(summary)

## 4. Interpretation and data requirements

A bullwhip ratio above 1 means that the variance of the relevant upstream order series exceeds the variance of customer demand over the same aligned periods. A defensible real-project analysis requires observed customer demand and order quantities for retailer, wholesaler and/or manufacturer stages. Do not use the synthetic values in the final report as evidence about a real supply chain. Document missing stages, aggregation frequency and sample size.